# Baselines — official implementations, one evaluation protocol
Each adapter (1) writes our per-entity data into the repository's expected layout, (2) trains with the **official code and
official hyper-parameters**, (3) extracts **per-point anomaly scores** for the training slice and the test stream, and
(4) evaluates them with the *same* protocol as M-TTA: unsupervised τ = 99th percentile of the method's own scores on the
last 10% of training data, point-wise F1/AUROC/AUPRC **without point adjustment**, all entities. Results are JSON files in
`results_gpu/baselines/<DS>/` with the same schema as the M-TTA runs so the final table is one `pd.concat`.

Repositories (cloned by cell 0): TranAD (TranAD, MTAD-GAT, LSTM-AD, USAD, OmniAnomaly, GDN) · Time-Series-Library (TimesNet) ·
Anomaly-Transformer · KDD2023-DCdetector · M2N2 (M2N2, THOC).

> Adapters were written against the repository code as of Sep 2026 and verified for structure, not executed end-to-end here.
> If a repository changes its argument names, the `# CHECK` comments mark the lines to reconcile.

In [ ]:
# --- 0. setup: clone repos, common evaluation ------------------------------------------------------
import os, sys, json, glob, subprocess, shutil, time, importlib, types, gc
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
ROOT = os.path.abspath(".").replace(os.sep, "/")
if "/repos/" in ROOT + "/": ROOT = ROOT.split("/repos/")[0]            # cwd was left inside a repo by a failed run
os.chdir(ROOT); assert os.path.isdir(f"{ROOT}/data"), f"run this notebook from the project folder (found {ROOT})"
REPOS = f"{ROOT}/repos"; os.makedirs(REPOS, exist_ok=True)
OUT_BASE = f"{ROOT}/results_gpu/baselines"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("device:", DEVICE)

REPO_URLS = {"TranAD": "https://github.com/imperial-qore/TranAD",
             "Time-Series-Library": "https://github.com/thuml/Time-Series-Library",
             "Anomaly-Transformer": "https://github.com/thuml/Anomaly-Transformer",
             "KDD2023-DCdetector": "https://github.com/DAMO-DI-ML/KDD2023-DCdetector",
             "M2N2": "https://github.com/carrtesy/M2N2",
             "mtad-gat-pytorch": "https://github.com/ML4ITS/mtad-gat-pytorch"}
import urllib.request, zipfile, io
REPO_BRANCH = {"TranAD": "main", "Time-Series-Library": "main", "Anomaly-Transformer": "main", "KDD2023-DCdetector": "main", "M2N2": "master", "mtad-gat-pytorch": "main"}
def fetch_repo(name, url):
    """Download the GitHub zip archive (no git needed) and unpack as repos/<name>."""
    dest = f"{REPOS}/{name}"
    if os.path.isdir(dest): return
    owner_repo = url.split("github.com/")[1]; br = REPO_BRANCH[name]
    zurl = f"https://codeload.github.com/{owner_repo}/zip/refs/heads/{br}"
    print("downloading", zurl); data = urllib.request.urlopen(zurl, timeout=120).read()
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        top = z.namelist()[0].split("/")[0]; z.extractall(REPOS)
    os.rename(f"{REPOS}/{top}", dest)
for name, url in REPO_URLS.items(): fetch_repo(name, url)
print("repos:", os.listdir(REPOS))

# ---- entities and our data (produced by MTTA_experiments.ipynb cell 2)
SMD_MACHINES = [f"machine-1-{i}" for i in range(1, 9)] + [f"machine-2-{i}" for i in range(1, 10)] + [f"machine-3-{i}" for i in range(1, 12)]
MSL_IDS = ["P-15","M-6","M-1","M-2","S-2","P-10","T-4","T-5","F-7","M-3","M-4","M-5","C-1","C-2","T-12","T-13","F-4","F-5","D-14","T-9","P-14","T-8","P-11","D-15","D-16","M-7","F-8"]
def load_entity(ds, ent):
    if ds == "SMD":
        tr = np.loadtxt(f"{ROOT}/data/SMD/train_{ent}.txt", delimiter=",", dtype=np.float32)
        te = np.loadtxt(f"{ROOT}/data/SMD/test_{ent}.txt", delimiter=",", dtype=np.float32)
        lb = np.loadtxt(f"{ROOT}/data/SMD/test_label_{ent}.txt", dtype=np.int64)
    else:
        tr = np.load(f"{ROOT}/data/{ds}/{ent}_train.npy").astype(np.float32); te = np.load(f"{ROOT}/data/{ds}/{ent}_test.npy").astype(np.float32)
        lb = np.load(f"{ROOT}/data/{ds}/{ent}_test_label.npy").astype(np.int64).reshape(-1)
    return tr, te, lb

# ---- identical evaluation protocol (copy of point_metrics in MTTA_experiments.ipynb)
def point_metrics(score, label, tau):
    from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
    if not (0 < label.sum() < len(label)): return {k: float("nan") for k in ("auroc", "auprc", "f1_unsup_tau", "f1_best")}
    pred = (score > tau).astype(int)
    tp = int(((pred == 1) & (label == 1)).sum()); fp = int(((pred == 1) & (label == 0)).sum()); fn = int(((pred == 0) & (label == 1)).sum())
    prec, rec = tp / max(1, tp + fp), tp / max(1, tp + fn); p, r, _ = precision_recall_curve(label, score)
    return {"auroc": float(roc_auc_score(label, score)), "auprc": float(average_precision_score(label, score)),
            "f1_unsup_tau": 2 * prec * rec / max(1e-12, prec + rec), "f1_best": float(np.max(2 * p * r / np.maximum(p + r, 1e-12)))}

def unsup_tau(train_scores, val_frac=0.1, q=0.99):
    """tau from the last val_frac of the method's OWN training scores (no test labels). Same rule as M-TTA (val_frac=0.1, q=0.99)."""
    n = max(1, int(len(train_scores) * val_frac)); return float(np.quantile(train_scores[-n:], q))

def align(score, label):
    """Some repos score floor(T/win)*win points; align by truncating labels (never pad scores)."""
    n = min(len(score), len(label)); return np.asarray(score[:n], float), np.asarray(label[:n], int)

def save_result(ds, method, ent, seed, train_scores, test_scores, label, extra=None):
    os.makedirs(f"{OUT_BASE}/{ds}", exist_ok=True)
    test_scores, label = align(test_scores, label); tau = unsup_tau(train_scores)
    res = {"tag": method, "dataset": ds, "train_entity": ent, "test_entity": ent, "seed": seed, "tau0": tau,
           "n_points_scored": int(len(test_scores)), **point_metrics(test_scores, label, tau), **(extra or {})}
    json.dump(res, open(f"{OUT_BASE}/{ds}/{method}__{ent}__{ent}__s{seed}.json", "w"))
    np.save(f"{OUT_BASE}/{ds}/{method}__{ent}__s{seed}_scores.npy", test_scores.astype(np.float32))
    print(f"{method:14s} {ds} {ent:12s} s{seed}  AUROC={res['auroc']:.3f} AUPRC={res['auprc']:.3f} F1@tau={res['f1_unsup_tau']:.3f} F1best={res['f1_best']:.3f}")
    return res

def use_repo(path):
    """Make `path` the active repository: put it first on sys.path, drop other repo paths, and purge cached modules that
    came from other repositories or share common top-level names (utils, models, exp, data, ...)."""
    path = os.path.abspath(path).replace(os.sep, "/")
    sys.path[:] = [q for q in sys.path if not os.path.abspath(q).replace(os.sep, "/").startswith(REPOS.replace(os.sep, "/") + "/")]
    sys.path.insert(0, path)
    common = ("utils", "models", "model", "exp", "data", "data_provider", "data_factory", "layers", "solver", "metrics", "src",
              "main", "mtad_gat", "training", "prediction", "modules", "eval_methods", "spot", "Exp", "cfgs", "args", "timefeatures")
    for name, mod in list(sys.modules.items()):
        top = name.split(".")[0]; f = (getattr(mod, "__file__", None) or "").replace(os.sep, "/")
        if top in common or (f and REPOS.replace(os.sep, "/") + "/" in f and not f.startswith(path + "/")):
            del sys.modules[name]
    os.chdir(path)

def done(ds, method, ent, seed): return os.path.exists(f"{OUT_BASE}/{ds}/{method}__{ent}__{ent}__s{seed}.json")

## 1. TranAD harness → TranAD, LSTM-AD, USAD, OmniAnomaly
`repos/TranAD/main.py` exposes `load_model`, `convert_to_windows`, `backprop` (returns per-point, per-dimension loss when
`training=False`). `dgl` is stubbed (only needed by the MTAD_GAT/GDN classes we do not use here). Their preprocessing is min–max with training statistics (`preprocess.py`); we apply the same.
Hyper-parameters: the repository defaults (`src/models.py`, `src/constants.py`).

In [ ]:
# --- dependency shims: src/models.py imports dgl (only used by MTAD_GAT/GDN, which we take from another repo);
#     src/plotting.py applies a SciencePlots matplotlib style. Neither is needed for TranAD/LSTM_AD/USAD/OmniAnomaly.
if "dgl" not in sys.modules:
    dgl = types.ModuleType("dgl"); dgl.nn = types.ModuleType("dgl.nn"); dgl.nn.GATConv = object
    dgl.graph = dgl.add_self_loop = lambda *a, **k: None; sys.modules["dgl"] = dgl; sys.modules["dgl.nn"] = dgl.nn
import matplotlib.pyplot as plt; plt.style.use = lambda *a, **k: None
sys.path.insert(0, f"{REPOS}/TranAD"); os.chdir(f"{REPOS}/TranAD")
sys.argv = ["main.py", "--dataset", "SMD", "--model", "TranAD"]        # parser needs argv; values are overridden below
import importlib; tranad_main = importlib.import_module("main"); os.chdir(ROOT)
from src.models import *   # noqa
# --- torch >= 2.1 compatibility: the repo's custom Transformer layers predate the is_causal/tgt_is_causal keyword arguments
import src.dlutils as _dl
#     (idempotent: the original forward is stored on the class once, so re-running this cell is safe)
if not hasattr(_dl.TransformerEncoderLayer, "_orig_forward"): _dl.TransformerEncoderLayer._orig_forward = _dl.TransformerEncoderLayer.forward
if not hasattr(_dl.TransformerDecoderLayer, "_orig_forward"): _dl.TransformerDecoderLayer._orig_forward = _dl.TransformerDecoderLayer.forward
def _enc_forward(self, src, src_mask=None, src_key_padding_mask=None, **kw):
    return _dl.TransformerEncoderLayer._orig_forward(self, src, src_mask, src_key_padding_mask)
def _dec_forward(self, tgt, memory, tgt_mask=None, memory_mask=None, tgt_key_padding_mask=None, memory_key_padding_mask=None, **kw):
    return _dl.TransformerDecoderLayer._orig_forward(self, tgt, memory, tgt_mask, memory_mask, tgt_key_padding_mask, memory_key_padding_mask)
_dl.TransformerEncoderLayer.forward = _enc_forward; _dl.TransformerDecoderLayer.forward = _dec_forward

def tranad_minmax(tr, te):                                             # preprocess.py normalize3
    mn, mx = tr.min(0), tr.max(0); d = np.where(mx - mn == 0, 1.0, mx - mn)
    return ((tr - mn) / d).astype(np.float32), ((te - mn) / d).astype(np.float32)

def run_tranad_model(ds, ent, method, seed, epochs=None):
    if done(ds, method, ent, seed): return
    torch.manual_seed(seed); np.random.seed(seed)
    tr, te, lb = load_entity(ds, ent); tr, te = tranad_minmax(tr, te)
    dims = tr.shape[1]
    model, optimizer, scheduler, _, _ = tranad_main.load_model(method, dims)          # fresh model (no checkpoint)
    model = model.to("cpu")                                                           # CHECK: repo trains on CPU by default; move to DEVICE if adapted
    trainD = torch.tensor(tr, dtype=torch.float64 if next(model.parameters()).dtype == torch.float64 else torch.float32)
    testD = torch.tensor(te, dtype=trainD.dtype)
    trainO, testO = trainD, testD
    tranad_main.args.model = method        # convert_to_windows flattens (or not) based on the global CLI arg, not on the model object
    if model.name in ["Attention", "DAGMM", "USAD", "MSCRED", "CAE_M", "GDN", "MTAD_GAT", "MAD_GAN"] or "TranAD" in model.name:
        trainD, testD = tranad_main.convert_to_windows(trainD, model), tranad_main.convert_to_windows(testD, model)
    E = epochs or 5                                                                    # repo default num_epochs=5
    for e in range(E):
        tranad_main.backprop(e, model, trainD, trainO, optimizer, scheduler, training=True)
    tr_loss, _ = tranad_main.backprop(0, model, trainD, trainO, optimizer, scheduler, training=False)
    te_loss, _ = tranad_main.backprop(0, model, testD, testO, optimizer, scheduler, training=False)
    tr_s = np.asarray(tr_loss).reshape(len(tr), -1).mean(1); te_s = np.asarray(te_loss).reshape(len(te), -1).mean(1)
    save_result(ds, method, ent, seed, tr_s, te_s, lb)

TRANAD_METHODS = ["TranAD", "LSTM_AD", "USAD", "OmniAnomaly"]      # MTAD-GAT: see the next cell (dgl-free repo)
SEEDS_BASE = list(range(10))
# for ds, ents in (("SMD", SMD_MACHINES), ("MSL", MSL_IDS)):
#     for seed in SEEDS_BASE:
#         for ent in tqdm(ents, desc=f"TranAD-harness {ds} s{seed}"):
#             for m in TRANAD_METHODS: run_tranad_model(ds, ent, m, seed)

## 1b. MTAD-GAT (ML4ITS/mtad-gat-pytorch, pure PyTorch, official default hyper-parameters)
`Predictor.get_score(values)` returns a per-point global anomaly score (`A_Score_Global`, forecast + reconstruction) for
points `window_size:`; labels are aligned the same way the repository does (`y_test[window_size:]`).

In [ ]:
MG = f"{REPOS}/mtad-gat-pytorch"
def run_mtad_gat(ds, ent, seed, epochs=30):
    if done(ds, "MTAD_GAT", ent, seed): return
    torch.manual_seed(seed); np.random.seed(seed)
    use_repo(MG)
    try:
        return _run_mtad_gat_body(ds, ent, seed, epochs)
    finally:
        os.chdir(ROOT)

def _run_mtad_gat_body(ds, ent, seed, epochs):
    from mtad_gat import MTAD_GAT; from training import Trainer; from prediction import Predictor
    from utils import SlidingWindowDataset, create_data_loaders
    tr, te, lb = load_entity(ds, ent)
    mn, mx = tr.min(0), tr.max(0); d = np.where(mx - mn == 0, 1.0, mx - mn); tr, te = (tr - mn) / d, (te - mn) / d   # repo: MinMax on train
    W, C = 100, tr.shape[1]                                                            # lookback=100 (repo default)
    train_ds = SlidingWindowDataset(tr, W, None); test_ds = SlidingWindowDataset(te, W, None)
    train_loader, val_loader, _ = create_data_loaders(train_ds, 256, 0.1, True, test_dataset=test_ds)
    model = MTAD_GAT(C, W, C, kernel_size=7, use_gatv2=True, feat_gat_embed_dim=None, time_gat_embed_dim=None, gru_n_layers=1, gru_hid_dim=150,
                     forecast_n_layers=3, forecast_hid_dim=150, recon_n_layers=1, recon_hid_dim=150, dropout=0.3, alpha=0.2)   # args.py defaults
    save_path = f"{ROOT}/checkpoints_mtadgat/{ds}_{ent}_s{seed}"; os.makedirs(save_path, exist_ok=True)
    trainer = Trainer(model, torch.optim.Adam(model.parameters(), lr=1e-3), W, C, None, epochs, 256, 1e-3, torch.nn.MSELoss(), torch.nn.MSELoss(),
                      DEVICE.type == "cuda", save_path, save_path, 1, False, "")
    trainer.fit(train_loader, val_loader)
    pred_args = {"dataset": ds, "target_dims": None, "scale_scores": False, "level": None, "q": None, "dynamic_pot": False,
                 "use_mov_av": False, "gamma": 1, "reg_level": 1, "save_path": save_path, "batch_size": 256, "use_cuda": DEVICE.type == "cuda"}   # CHECK keys vs train.py prediction_args
    predictor = Predictor(trainer.model, W, C, pred_args)
    tr_t, te_t = torch.tensor(tr, dtype=torch.float32), torch.tensor(te, dtype=torch.float32)     # get_score expects tensors
    tr_s = predictor.get_score(tr_t)["A_Score_Global"].values; te_s = predictor.get_score(te_t)["A_Score_Global"].values
    save_result(ds, "MTAD_GAT", ent, seed, tr_s, te_s, lb[W:])          # labels aligned as in the repo
    del model, trainer, predictor; gc.collect(); torch.cuda.empty_cache()

# for ds, ents in (("SMD", SMD_MACHINES), ("MSL", MSL_IDS)):
#     for seed in SEEDS_BASE:
#         for ent in tqdm(ents, desc=f"MTAD-GAT {ds} s{seed}"): run_mtad_gat(ds, ent, seed)

## 2. Time-Series-Library → TimesNet
Per entity we create a dataset folder in the library's SMD/MSL layout (`<DS>_train.npy`, `<DS>_test.npy`, `<DS>_test_label.npy`;
the loader standardizes internally), train with the official script arguments, then compute per-point energies exactly as
`Exp_Anomaly_Detection.test` does (mean MSE over channels) for the train slice and the test set.

In [ ]:
TSL = f"{REPOS}/Time-Series-Library"
def tsl_export(ds, ent):
    tr, te, lb = load_entity(ds, ent); d = f"{ROOT}/data/tsl/{ds}_{ent}"; os.makedirs(d, exist_ok=True)
    np.save(f"{d}/{ds}_train.npy", tr); np.save(f"{d}/{ds}_test.npy", te); np.save(f"{d}/{ds}_test_label.npy", lb)
    return d, tr.shape[1], lb

def tsl_args(ds, ent, root, C, seed):
    from argparse import Namespace
    a = Namespace(task_name="anomaly_detection", is_training=1, model_id=f"{ds}_{ent}_s{seed}", model="TimesNet", data=ds, root_path=root,
                  data_path="", features="M", seq_len=100, label_len=0, pred_len=0, d_model=64, d_ff=64, e_layers=2, d_layers=1, enc_in=C, dec_in=C,
                  c_out=C, top_k=5, num_kernels=6, anomaly_ratio=0.5, batch_size=128, train_epochs=3, learning_rate=1e-4, patience=3, des="Exp",
                  itr=1, num_workers=0, use_gpu=DEVICE.type == "cuda", gpu=0, gpu_type="cuda", use_multi_gpu=False, devices="0", checkpoints=f"{ROOT}/checkpoints_tsl",
                  embed="timeF", freq="h", dropout=0.1, activation="gelu", factor=1, n_heads=8, moving_avg=25, distil=True, output_attention=False,
                  seasonal_patterns="Monthly", inverse=False, use_dtw=False, augmentation_ratio=0, loss="MSE", lradj="type1", use_amp=False,
                  expand=2, d_conv=4, channel_independence=0, decomp_method="moving_avg", use_norm=1, down_sampling_layers=0, down_sampling_window=1,
                  down_sampling_method=None, seg_len=96)   # CHECK: mirror scripts/anomaly_detection/SMD/TimesNet.sh + run.py defaults for your library version
    return a

@torch.no_grad()
def tsl_energy(exp, flag):
    """Replicates the scoring loop of exp_anomaly_detection.test(): per-window MSE averaged over channels -> per point."""
    loader = exp._get_data(flag=flag)[1]; crit = torch.nn.MSELoss(reduce=False); exp.model.eval(); out = []
    for batch_x, _ in loader:
        batch_x = batch_x.float().to(exp.device); rec = exp.model(batch_x, None, None, None)
        out.append(torch.mean(crit(batch_x, rec), dim=-1).detach().cpu().numpy())
    return np.concatenate(out, 0).reshape(-1)

def run_timesnet(ds, ent, seed):
    if done(ds, "TimesNet", ent, seed): return
    torch.manual_seed(seed); np.random.seed(seed)
    root, C, lb = tsl_export(ds, ent); args = tsl_args(ds, ent, root, C, seed)
    use_repo(TSL)
    try:
        from exp.exp_anomaly_detection import Exp_Anomaly_Detection
        exp = Exp_Anomaly_Detection(args); setting = f"{args.model_id}_{args.model}"
        exp.train(setting)                                                   # official training loop (early stopping on val)
        tr_s = tsl_energy(exp, "train"); te_s = tsl_energy(exp, "test")     # 'test' flag = non-overlapping windows over the test set
    finally:
        os.chdir(ROOT)
    save_result(ds, "TimesNet", ent, seed, tr_s, te_s, lb)
    del exp; gc.collect(); torch.cuda.empty_cache()

# for ds, ents in (("SMD", SMD_MACHINES), ("MSL", MSL_IDS)):
#     for seed in SEEDS_BASE:
#         for ent in tqdm(ents, desc=f"TimesNet {ds} s{seed}"): run_timesnet(ds, ent, seed)

## 3. Anomaly-Transformer and DCdetector
Both repositories share the `Solver(config)` design and compute per-point energies inside `test()` before thresholding.
We call the official `train()`, then replicate the energy loop (Anomaly-Transformer: association discrepancy × reconstruction
error with temperature 50; DCdetector: association discrepancy only) on the `train` and `thre` loaders.

In [ ]:
def solver_energy(solver, loader, kind, win_size):
    """kind: 'AT' -> metric * rec_loss ; 'DC' -> metric only. Mirrors the repos' test() loops."""
    import torch.nn.functional as F
    from solver import my_kl_loss                            # defined in solver.py in both repositories (verified)
    solver.model.eval(); temperature = 50; crit = torch.nn.MSELoss(reduce=False); out = []
    for batch in loader:
        x = batch[0].float().to(solver.device if hasattr(solver, "device") else DEVICE)
        if kind == "AT":
            output, series, prior, _ = solver.model(x); loss = torch.mean(crit(x, output), dim=-1)
        else:
            series, prior = solver.model(x); loss = None
        series_loss = prior_loss = 0.0
        for u in range(len(prior)):
            pn = prior[u] / torch.unsqueeze(torch.sum(prior[u], dim=-1), dim=-1).repeat(1, 1, 1, win_size)
            series_loss = series_loss + my_kl_loss(series[u], pn.detach()) * temperature
            prior_loss = prior_loss + my_kl_loss(pn, series[u].detach()) * temperature
        metric = torch.softmax((-series_loss - prior_loss), dim=-1)
        cri = (metric * loss) if kind == "AT" else metric
        out.append(cri.detach().cpu().numpy())
    return np.concatenate(out, 0).reshape(-1)

def run_solver_repo(repo, kind, ds, ent, seed, method):
    if done(ds, method, ent, seed): return
    torch.manual_seed(seed); np.random.seed(seed)
    root, C, lb = tsl_export(ds, ent)                       # same per-entity layout (<DS>_train.npy ...) works for both repos
    rp = f"{REPOS}/{repo}"; use_repo(rp)
    from solver import Solver
    from data_factory.data_loader import get_loader_segment
    cfg = dict(lr=1e-4, num_epochs=3, k=3, win_size=100, input_c=C, output_c=C, batch_size=128, pretrained_model=None, dataset=ds,
               mode="train", data_path=root, model_save_path=f"{ROOT}/checkpoints_{method}", anormly_ratio=0.5, index=0)
    if kind == "DC":   # DCdetector main.py argument set (verified): patch_size is a list of patch lengths that must divide win_size
        cfg.update(dict(patch_size=[5, 10], loss_fuc="MSE", n_heads=1, e_layers=3, d_model=256, rec_timeseries=1,
                        use_gpu=DEVICE.type == "cuda", gpu=0, use_multi_gpu=False, devices="0"))     # CHECK d_model/e_layers vs scripts/*.sh
    os.makedirs(cfg["model_save_path"], exist_ok=True)
    try:
        solver = Solver(cfg); solver.train()
        tr_loader = get_loader_segment(root, batch_size=128, win_size=100, step=100, mode="train", dataset=ds)
        th_loader = get_loader_segment(root, batch_size=128, win_size=100, step=100, mode="thre", dataset=ds)
        tr_s = solver_energy(solver, tr_loader, kind, 100); te_s = solver_energy(solver, th_loader, kind, 100)
    finally:
        os.chdir(ROOT)
    save_result(ds, method, ent, seed, tr_s, te_s, lb)
    del solver; gc.collect(); torch.cuda.empty_cache()

# for ds, ents in (("SMD", SMD_MACHINES), ("MSL", MSL_IDS)):
#     for seed in SEEDS_BASE:
#         for ent in tqdm(ents, desc=f"AT/DC {ds} s{seed}"):
#             run_solver_repo("Anomaly-Transformer", "AT", ds, ent, seed, "AnomalyTransformer")
#             run_solver_repo("KDD2023-DCdetector", "DC", ds, ent, seed, "DCdetector")

## 4. M2N2 repository → M2N2 (MLP + Detrend + test-time adaptation), THOC
Training uses the official `train.py` (Hydra) with the SMD/MSL settings from `scripts/MLP/train/*.sh` (window 5, stride 5, batch 8,
latent 16, normalization=Detrend). Scores are harvested through the official `Tester`: `prepare_stats()` gives offline per-point
scores, `online()` gives the adapted-stream scores. **The online initial threshold is the unsupervised τ (train-score quantile)
instead of the repository's label-based `th_off_f1_best`**, so the comparison follows the same protocol as M-TTA. wandb is disabled.
`M2N2_offline` (no adaptation) is saved as a by-product.

In [ ]:
M2 = f"{REPOS}/M2N2"
# --- one-time repo fixes: utils/secret.py ships as invalid Python (placeholders), and train/test call wandb -> disable it
open(f"{M2}/utils/secret.py", "w").write('WANDB_API_KEY = "0" * 40\nWANDB_PROJECT_NAME = "m2n2_local"\nWANDB_ENTITY = "local"\n')
os.environ["WANDB_MODE"] = "disabled"

def m2n2_prepare_data():
    """Copy our raw files into the layout read by repos/M2N2/data/load_data.py (verified): data/SMD/{train,test,test_label}/<m>.txt,
    data/MSL/{train,test}/<id>.npy + data/MSL/labeled_anomalies.csv"""
    for sub in ("train", "test", "test_label"): os.makedirs(f"{M2}/data/SMD/{sub}", exist_ok=True)
    for m in SMD_MACHINES:
        for sub in ("train", "test", "test_label"): shutil.copy(f"{ROOT}/data/SMD/{sub}_{m}.txt", f"{M2}/data/SMD/{sub}/{m}.txt")
    for sub in ("train", "test"): os.makedirs(f"{M2}/data/MSL/{sub}", exist_ok=True)
    for cid in MSL_IDS:
        np.save(f"{M2}/data/MSL/train/{cid}.npy", np.load(f"{ROOT}/data/MSL/{cid}_train.npy")); np.save(f"{M2}/data/MSL/test/{cid}.npy", np.load(f"{ROOT}/data/MSL/{cid}_test.npy"))
    import zipfile
    with zipfile.ZipFile(f"{ROOT}/SML__MSL.zip") as z: open(f"{M2}/data/MSL/labeled_anomalies.csv", "wb").write(z.read("labeled_anomalies.csv"))
    print("M2N2 data ready")

# official SMD/MSL settings from scripts/MLP/train/*.sh ; normalization=Detrend is the M2N2 method, None = plain MLP+TTA
def m2n2_overrides(ds, ent, seed, model, normalization):
    ov = [f"SEED={seed}", f"dataset={ds}", f"+dataset_id={ent}", "window_size=5", "stride=5", "eval_stride=5", "batch_size=8", "log_freq=1",
          f"normalization={normalization}", f"model={model}"]
    if model == "MLP": ov.append("model.latent_dim=16")
    return ov

def run_m2n2(ds, ent, seed, model="MLP", normalization="Detrend", method="M2N2"):
    """Train with the official train.py (subprocess), then harvest per-point scores through the official Tester."""
    if done(ds, method, ent, seed): return
    ov = m2n2_overrides(ds, ent, seed, model, normalization)
    subprocess.run([sys.executable, "train.py"] + ov, cwd=M2, check=True, env={**os.environ, "WANDB_MODE": "disabled"})
    # ---- harvest (mirrors test.py without wandb)
    use_repo(M2)
    try:
        return _m2n2_harvest(ds, ent, seed, model, normalization, method, ov)
    finally:
        os.chdir(ROOT)

def _m2n2_harvest(ds, ent, seed, model, normalization, method, ov):
    from hydra import initialize_config_dir, compose
    from hydra.core.global_hydra import GlobalHydra
    from utils.argpass import prepare_arguments; from utils.logger import make_logger; from utils.tools import SEED_everything
    from data.load_data import DataFactory
    from Exp import MLP_Tester, THOC_Tester
    import wandb; wandb.init(mode="disabled")
    GlobalHydra.instance().clear()
    with initialize_config_dir(config_dir=f"{M2}/cfgs", version_base=None):
        cfg = compose(config_name="test_defaults", overrides=ov)
    args = prepare_arguments(cfg); logger = make_logger(os.path.join(args.log_path, f"{args.exp_id}_harvest.log"))
    SEED_everything(args.SEED)
    train_dataset, train_loader, test_dataset, test_loader = DataFactory(args, logger)()
    args.num_channels = train_dataset.X.shape[1]
    if model == "THOC":
        # THOC scores one value per WINDOW; the repo's shared prepare_stats() assumes per-point scores and crashes.
        # Bypass it: load the checkpoint, score windows, and expand each window score to its window_size points
        # (eval_stride == window_size -> non-overlapping windows, so the expansion is exact).
        tester = THOC_Tester(args=args, logger=logger, train_loader=train_loader, test_loader=test_loader, load=False)
        tester.load_trained_model()
        W = int(args.window_size)
        tr_s = np.repeat(np.asarray(tester.calculate_anomaly_scores(train_loader)).reshape(-1), W)
        te_off = np.repeat(np.asarray(tester.calculate_anomaly_scores(test_loader)).reshape(-1), W)
    else:
        tester = MLP_Tester(args=args, logger=logger, train_loader=train_loader, test_loader=test_loader, load=True)
        tester.prepare_stats()                                                # computes train_anoscs / test_anoscs (offline)
        tr_s = np.asarray(tester.train_anoscs).reshape(-1) if hasattr(tester, "train_anoscs") else np.asarray(tester.calculate_anomaly_scores(train_loader)).reshape(-1)
        te_off = np.asarray(tester.test_anoscs).reshape(-1)
    tr_s = tr_s[tr_s >= 0]                                                    # -1 entries (if any) are padding
    if model == "MLP":                                                        # M2N2: online adaptation with an UNSUPERVISED initial threshold
        init_thr = unsup_tau(tr_s)                                            # NOT th_off_f1_best (label-based) -> same rule as M-TTA
        out = tester.online(test_loader, init_thr, normalization=args.normalization)
        te_on = np.asarray(out[0] if isinstance(out, (tuple, list)) else out).reshape(-1)
    else:
        te_on = None
    lb = load_entity(ds, ent)[2]
    save_result(ds, method, ent, seed, tr_s, te_on if te_on is not None else te_off, lb, extra={"model": model, "normalization": normalization})
    if te_on is not None: save_result(ds, method + "_offline", ent, seed, tr_s, te_off, lb, extra={"model": model, "normalization": normalization})

# m2n2_prepare_data()
# for ds, ents in (("SMD", SMD_MACHINES), ("MSL", MSL_IDS)):
#     for seed in SEEDS_BASE:
#         for ent in tqdm(ents, desc=f"M2N2/THOC {ds} s{seed}"):
#             run_m2n2(ds, ent, seed, model="MLP", normalization="Detrend", method="M2N2")
#             run_m2n2(ds, ent, seed, model="THOC", normalization="None", method="THOC")

## 5. Combined table (baselines + M-TTA variants)

In [ ]:
def load_dir(d):
    rows = []
    for f in glob.glob(f"{d}/*.json"):
        r = json.load(open(f)); r.pop("segments", None); rows.append(r)
    return pd.DataFrame(rows)
pd.set_option("display.width", 220); pd.set_option("display.precision", 3)
def combined_table(ds):
    ours = load_dir(f"{ROOT}/results_gpu/{ds}"); ours = ours[ours.train_entity == ours.test_entity]
    base = load_dir(f"{OUT_BASE}/{ds}")
    df = pd.concat([ours, base], ignore_index=True)
    # our fused-score variant reported as its own row
    fused = ours[ours.tag == "gate_gbuf5k"].copy(); fused["tag"] = "M-TTA (u-fused score)"; fused["auroc"] = fused["auroc_fused_2.0"]; fused["auprc"] = fused["auprc_fused_2.0"]
    df = pd.concat([df, fused], ignore_index=True)
    ent = df.groupby(["tag", "train_entity"])[["auroc", "auprc", "f1_unsup_tau", "f1_best"]].mean().reset_index()
    tab = ent.groupby("tag")[["auroc", "auprc", "f1_unsup_tau", "f1_best"]].agg(["mean", "std"])
    tab["n_entities"] = ent.groupby("tag").size(); tab["n_seeds"] = df.groupby("tag").seed.nunique()
    print(f"=== {ds}: mean over entities (seed-averaged) ± std over entities ==="); print(tab.sort_values(("auroc", "mean"), ascending=False).to_string())
    return tab
# tab_smd = combined_table("SMD"); tab_msl = combined_table("MSL")